# Contrato temporal e divisão para modelagem

O dataset registra pedidos entre setembro de 2016 e agosto de 2018. Como queremos simular uma decisão operacional, os modelos não poderão aprender com o futuro para prever o passado. A divisão principal será cronológica, preservando a ordem de compra.

## Cobertura temporal observada

A data de referência para ordenar as observações é `order_purchase_timestamp`, pois representa o momento em que o pedido entrou no sistema. A tabela mensal mostra o volume e a participação acumulada antes de escolher os cortes.

Os primeiros meses têm pouco volume, enquanto 2017 e 2018 concentram a massa principal. Isso recomenda cortes em blocos de calendário, e não uma divisão aleatória.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

LANG = "pt"
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "data").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from ecom_delivery_sla import (
    build_candidate_order_features,
    build_supervised_orders,
    plot_temporal_split,
    read_olist_tables,
    temporal_monthly_summary,
    temporal_split,
    temporal_split_summary,
)

tables = read_olist_tables(
    project_root / "data" / "raw" / "olist" / "csv",
    table_names=("orders", "items", "products", "payments", "customers", "sellers", "geolocation"),
)
supervised = build_supervised_orders(tables["orders"])
candidate_features = build_candidate_order_features(
    orders=supervised,
    items=tables["items"],
    products=tables["products"],
    payments=tables["payments"],
    customers=tables["customers"],
    sellers=tables["sellers"],
    geolocation=tables["geolocation"],
)

monthly = temporal_monthly_summary(candidate_features)
display(monthly)
print(f"Orders available for temporal analysis: {len(candidate_features):,}")

,purchase_month,orders,cumulative_share
0,2016-09,1,0.000010
1,2016-10,265,0.002757
2,2016-12,1,0.002768
3,2017-01,750,0.010542
4,2017-02,1653,0.027677
5,2017-03,2546,0.054069
6,2017-04,2303,0.077941
7,2017-05,3545,0.114689
8,2017-06,3135,0.147186
9,2017-07,3872,0.187322


Orders available for temporal analysis: 96,470


### Insight

A população candidata mantém a mesma janela temporal da população supervisionada e pode ser ordenada pela compra. A ausência de meses em alguns períodos não será preenchida artificialmente; ela faz parte da cobertura histórica observada.

## Cortes temporais candidatos

Como primeiro desenho, usamos o fim de março de 2018 como limite de treino e o fim de junho de 2018 como limite de validação. Assim, o teste fica reservado para julho e agosto de 2018. Esses cortes são candidatos para a modelagem e ainda podem ser revisados após a auditoria de disponibilidade das features.

### Insight

O teste permanece cronologicamente posterior ao treino e à validação. Nenhuma linha será embaralhada entre partições.

In [2]:
train_end = "2018-04-01"
validation_end = "2018-07-01"
splits = temporal_split(
    candidate_features,
    timestamp_col="order_purchase_timestamp",
    train_end=train_end,
    validation_end=validation_end,
)
display(temporal_split_summary(splits))
display(plot_temporal_split(splits, language=LANG))
print(f"Train end: {train_end}; validation end: {validation_end}")

,split,orders,share,first_purchase,last_purchase
0,train,64320,0.666736,2016-09-15 12:16:38,2018-03-31 23:54:10
1,validation,19643,0.203618,2018-04-01 00:11:32,2018-06-30 23:59:49
2,test,12507,0.129647,2018-07-01 00:25:07,2018-08-29 15:00:37


Train end: 2018-04-01; validation end: 2018-07-01


### Insight

A divisão temporal fornece uma simulação mais próxima do uso operacional: o modelo aprende com pedidos antigos, é ajustado em um período posterior e só então é avaliado no período mais recente. O teste não será consultado para escolher features ou hiperparâmetros.

## Folds expanding-window de desenvolvimento

Além do corte único, a seleção de modelos usa folds mensais expansivos (Tashman, 2000; Bergmeir e Benítez, 2012) de outubro/2017 a junho/2018, com embargo de 30 dias (López de Prado, 2018) entre o fim do treino e o início da validação. A tabela resume, por fold, o período, o número de pedidos de treino e validação e a prevalência de violação de SLA. Pedidos de julho–agosto/2018 não entram em nenhum fold. O gráfico a seguir apresenta, para cada fold, a linha do tempo com os períodos de treino, de embargo e de validação e, ao lado, a prevalência de treino e de validação.

In [3]:
from ecom_delivery_sla import (
    build_modeling_frame,
    expanding_fold_summary,
    expanding_window_folds,
)

modeling_frame = build_modeling_frame(candidate_features, supervised)
folds = expanding_window_folds(
    modeling_frame,
    timestamp_col="order_purchase_timestamp",
    first_val_month="2017-10",
    last_val_month="2018-06",
    embargo_days=30,
)
display(expanding_fold_summary(folds))

from ecom_delivery_sla_viz import plot_expanding_folds

display(plot_expanding_folds(expanding_fold_summary(folds), language=LANG))

,fold,train_start,train_end,validation_start,validation_end,train_orders,validation_orders,train_prevalence,validation_prevalence
0,2017-10,2016-09-15 12:16:38,2017-08-31 23:37:47,2017-10-01 00:03:33,2017-10-31 23:50:58,22264,4478,0.034675,0.041760
1,2017-11,2016-09-15 12:16:38,2017-10-01 23:41:19,2017-11-01 00:12:34,2017-11-30 23:36:03,26538,7288,0.036061,0.124040
2,2017-12,2016-09-15 12:16:38,2017-10-31 23:50:58,2017-12-01 00:01:14,2017-12-31 23:29:31,30892,5513,0.036935,0.074551
3,2018-01,2016-09-15 12:16:38,2017-12-01 23:54:03,2018-01-01 02:48:41,2018-01-31 23:58:22,38447,7069,0.053944,0.057009
4,2018-02,2016-09-15 12:16:38,2018-01-01 23:50:59,2018-02-01 00:24:55,2018-02-28 23:57:55,43765,6555,0.056186,0.141266
5,2018-03,2016-09-15 12:16:38,2018-01-29 23:50:39,2018-03-01 00:00:00,2018-03-31 23:54:10,50263,7003,0.056165,0.189633
6,2018-04,2016-09-15 12:16:38,2018-03-01 23:56:42,2018-04-01 00:11:32,2018-04-30 23:47:26,57585,6798,0.066962,0.045013
7,2018-05,2016-09-15 12:16:38,2018-03-31 23:54:10,2018-05-01 00:02:11,2018-05-31 23:51:24,64320,6749,0.079493,0.065639
8,2018-06,2016-09-15 12:16:38,2018-05-01 23:48:26,2018-06-01 00:39:55,2018-06-30 23:59:49,71370,6096,0.076054,0.011647


### Insight

Os folds crescem de forma monotônica no treino e cada validação é um único mês posterior ao treino mais o embargo. A coluna de prevalência mostra se a taxa de violação é estável entre folds; variações grandes indicam que a comparação entre meses deve considerar a sazonalidade e não apenas o desempenho médio. Os valores estão na tabela acima. O gráfico permite conferir visualmente o embargo entre o fim do treino e o início da validação.

## Deriva das features entre treino e validação

O PSI (population stability index) compara a distribuição de cada feature numérica no treino e na validação do último fold (Siddiqi, 2006; Yurdakul, 2018). Ele é descritivo: indica quais variáveis mudaram de distribuição, não quais prejudicam o modelo. O prazo prometido (`promised_lead_days`) é destacado porque é o principal sinal conhecido na compra. O gráfico seguinte mostra o PSI de todas as features numéricas. O eixo horizontal está em escala logarítmica porque os valores ocupam várias ordens de grandeza, de quase zero nas features estáveis até valores muito altos nas de calendário. Em escala linear, as features de PSI baixo ficariam comprimidas junto ao zero, e só as maiores barras seriam legíveis. Na escala logarítmica, cada marca principal do eixo vale dez vezes a anterior, e por isso o comprimento das barras não é proporcional ao PSI: a leitura deve ser feita pela posição do fim de cada barra no eixo, e não pela comparação de comprimentos.

As duas linhas verticais marcam limiares convencionais da prática de risco de crédito, área em que o PSI se originou. Abaixo de 0,10, a distribuição é tratada como estável; entre 0,10 e 0,25, como mudança moderada; acima de 0,25, como mudança relevante. Esses limiares não são regras deste projeto e servem apenas como referência de leitura. As features de calendário são apresentadas em cinza porque o PSI delas é alto por construção: a validação cobre um único mês, enquanto o treino cobre todos os meses anteriores, de modo que o mês da compra e as distâncias até o Natal e até a Black Friday assumem na validação valores concentrados, necessariamente diferentes dos do treino.

In [4]:
from ecom_delivery_sla import baseline_feature_catalog, feature_psi_table

feature_catalog_table = baseline_feature_catalog()
numeric_features = feature_catalog_table.loc[
    feature_catalog_table["feature_type"] == "numeric", "feature"
].tolist()
last_fold = folds[-1]
psi_table = feature_psi_table(
    last_fold["train"], last_fold["validation"], columns=numeric_features
)
display(psi_table.head(10))
display(psi_table[psi_table["feature"] == "promised_lead_days"])

from ecom_delivery_sla_viz import plot_psi

calendar_features = ["purchase_month", "days_to_black_friday", "days_to_christmas"]
display(plot_psi(psi_table, neutral_features=calendar_features, language=LANG))

,feature,psi
0,purchase_month,12.655842
1,days_to_christmas,12.440643
2,days_to_black_friday,12.434663
3,dest_hist_breach_rate,8.245999
4,dest_hist_median_lead_days,4.871000
5,seller_hist_breach_rate,2.659370
6,dest_in_degree,2.359899
7,edge_hist_breach_rate,1.543024
8,total_freight_value,0.313508
9,promised_lead_days,0.263818


,feature,psi
9,promised_lead_days,0.263818


### Insight

As features no topo da tabela são as que mais mudaram entre treino e validação e merecem checagem antes de qualquer interpretação de importância. A linha de `promised_lead_days` mostra a deriva do prazo prometido isoladamente. Valores de PSI usuais de referência (por exemplo, acima de 0,25 para mudança relevante) são convenções, não regras deste projeto, e os valores observados estão nas tabelas acima. No gráfico, além das features de calendário, ficam acima de 0,25 as taxas históricas de violação da rede (destino, vendedor e aresta), a mediana histórica de prazo do destino, o grau de entrada do destino, o valor total do frete e, por pouco, o prazo prometido. A deriva das features de rede é compatível com a forma como são construídas, porque acumulam histórico ao longo do tempo e mudam de distribuição à medida que a rede cresce.

## Sensibilidade à censura à direita (jul–ago/2018)

Pedidos enviados em julho–agosto/2018 que ainda não têm data de entrega são censurados (Klein e Moeschberger, 2003): o desfecho não foi observado. A tabela compara dois cenários descritivos: `exclude` (cenário principal, apenas pedidos entregues) e `include_as_breach` (limite superior, que conta todo censurado como violação). Ela traz apenas contagens e prevalência; nenhuma métrica de modelo é calculada nesse período.

In [5]:
from ecom_delivery_sla import right_censoring_sensitivity

display(right_censoring_sensitivity(tables["orders"]))

,scenario,supervised_orders,breaches,censored_orders,prevalence
0,exclude,12507,601,107,0.048053
1,include_as_breach,12614,708,107,0.056128


### Insight

A diferença de prevalência entre os cenários mede o quanto a censura pode distorcer o rótulo no fim da série. O cenário `include_as_breach` é um limite superior, pois nem todo pedido censurado violaria o prazo. Foi aprovado que esses pedidos são excluídos da população supervisionada no cenário principal (P-02 em `docs/REGISTRO-DE-DECISOES.md`); o cenário `include_as_breach` é mantido apenas como sensibilidade descritiva e será citado como limitação na avaliação final. O período segue fora da seleção de modelos.

## Disponibilidade temporal das features

A divisão cronológica sozinha não elimina vazamento. Cada feature ainda precisa ser conhecida no momento da compra. Indicadores históricos de vendedor, por exemplo, deverão ser recalculados dentro de cada janela usando somente pedidos anteriores. Valores imputados também serão ajustados exclusivamente no treino.

### Insight final

O contrato temporal está implementado como divisão testável, mas os cortes só serão congelados quando a disponibilidade das features e a política para sinais históricos estiverem aprovadas.

## Referências

Bergmeir, C. e Benítez, J. M. (2012). On the use of cross-validation for time series predictor evaluation. *Information Sciences*, 191, 192–213. https://doi.org/10.1016/j.ins.2011.12.028

Klein, J. P. e Moeschberger, M. L. (2003). *Survival Analysis: Techniques for Censored and Truncated Data* (2ª ed.). Springer.

López de Prado, M. (2018). *Advances in Financial Machine Learning*. Wiley.

Siddiqi, N. (2006). *Credit Risk Scorecards: Developing and Implementing Intelligent Credit Scoring*. Wiley.

Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: an analysis and review. *International Journal of Forecasting*, 16(4), 437–450. https://doi.org/10.1016/S0169-2070(00)00065-0

Yurdakul, B. (2018). *Statistical Properties of Population Stability Index*. Tese de doutorado, Western Michigan University. https://scholarworks.wmich.edu/dissertations/3208/